In [33]:
import pandas as pd
from pyproj import Proj, transform
from geopy.geocoders import Nominatim
import pyproj
import time
import re
import numpy as np
from tqdm import tqdm
from geopy.exc import GeocoderTimedOut
import math

In [34]:
#Logradouro
log = pd.read_csv("logradouro.csv", encoding='latin1', delimiter='\t')
log.columns



Index(['Data de Solicitação', 'Número da OS', 'Endereço', 'Serviço',
       'Setor Execução', 'Referência de Localização', 'Bairro', 'Município',
       'Funcionário / Equipe', 'Data de Execução', 'Data de Encerramento',
       'Quantidade de Serviço Executado', 'Usuário Solicitante', 'Situação',
       'Parecer Solicitação', 'Parecer Não Execução', 'Motivo Não Execução',
       'Data Limite de Execução', 'Parecer Execução', 'Data de Cancelamento',
       'Parecer Cancelamento', 'Data Final da Suspensão por Controle',
       'Motivo da Suspensão por Controle', 'Parecer da Suspensão por Controle',
       'Período de Suspensão por Controle (dias - horas:min:seg)',
       'Data Final da Liberação', 'Parecer da Liberação'],
      dtype='object')

In [35]:
pd.set_option('display.max_rows', None)
log['Serviço'].value_counts()

Serviço
RA - Consertar Ramal                                  144
RA - Consertar Rede                                    58
LA - Conserto Vazamento (Unidade Nao Identificada)     55
OP - Conserto de Rua - Asfalto                         14
instalacao de esperas de agua                          13
RA -  Verificar Irregularidade                          5
RA - Ampliacao de Rede                                  4
RETIRADA DE RAMAL / ESPERA                              3
INTERLIGACAO DE REDE LOTEAMENTO                         3
RA - Deslocamento/Remanejamento de Rede                 3
REATIVACAO DE REDE                                      1
INSTALACAO DE VENTOSA                                   1
VERIFICAR USO DE AGUA DIRETO DA REDE                    1
VAZAMENTO COLAR DE LIGACAO                              1
RA - Verificar Pressao d'Agua                           1
Name: count, dtype: int64

In [36]:
mapeamento_categorias = {
    'LA - Conserto Vazamento (Unidade Nao Identificada)': 'Vazamento',
}
pd.set_option('display.max_rows', 10)

# Aplique o mapeamento para criar a coluna 'Categoria de serviço'
log['Categoria de serviço'] = log['Serviço'].map(mapeamento_categorias)
log = log[log['Categoria de serviço']=='Vazamento']

log['Endereço'] = log['Endereço'].str.rstrip()
log['Endereço'] = log['Endereço'].apply(lambda x: x.rstrip('-') if x.endswith('-') else x)
log['Endereço'] = log['Endereço'].str.replace('Serv.', 'Servidão')
log['Endereço'] = log['Endereço'].str.replace('R.', 'Rua')
log['Endereço'] = log['Endereço'].apply(lambda x: re.sub(r'\([^()]*\)', '', x))
log['Endereço'] = log['Endereço'].str.title()
log

,Data de Solicitação,Número da OS,Endereço,Serviço,Setor Execução,Referência de Localização,Bairro,Município,Funcionário / Equipe,Data de Execução,...,Parecer Execução,Data de Cancelamento,Parecer Cancelamento,Data Final da Suspensão por Controle,Motivo da Suspensão por Controle,Parecer da Suspensão por Controle,Período de Suspensão por Controle (dias - horas:min:seg),Data Final da Liberação,Parecer da Liberação,Categoria de serviço
2,02/01/2023 08:36:00,115750,Rua Mal. Castelo Branco,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Bracinho,SCHROEDER,1001 - Operacional,09/01/2023 10:35:00,...,Foi efetuado o servico pelo Fernando e Dorival.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
7,06/01/2023 08:19:00,115806,Rua Barao Do Rio Branco,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Centro Leste,SCHROEDER,1001 - Operacional,08/01/2023 09:30:00,...,Foi efetuada a troca de motor e conserto no va...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
16,10/01/2023 10:08:00,116081,Rua Guilherme Piske,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Centro Norte,SCHROEDER,1001 - Operacional,11/01/2023 07:50:00,...,foi efetuado o servico pelo Wagner.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
17,10/01/2023 14:36:00,116097,Rua Henrique Ziebel,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Rio Hern,SCHROEDER,1001 - Operacional,17/01/2023 08:30:00,...,Foi efetuado o servico pelo Fernando e Wagner.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
25,16/01/2023 13:33:00,116221,Rua Jorge Lacerda,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Centro Norte,SCHROEDER,1001 - Operacional,17/01/2023 11:40:00,...,Foi efetuado o servico pelo Fernando e Wagner.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,09/10/2023 08:15:00,194506,Rua Barao Do Rio Branco,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Centro Leste,SCHROEDER,1001 - Operacional,18/10/2023 14:50:00,...,Executado por Adriano e Muka era um vazament...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
291,06/11/2023 08:42:00,196009,Rua Otto Elert,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Itoupava Acu,SCHROEDER,1001 - Operacional,06/11/2023 08:42:00,...,Executado por Muka e Adriano ) 06/10/2023 das ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
292,06/11/2023 08:47:00,196010,Rua Henrique Ziebel,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Rio Hern,SCHROEDER,1001 - Operacional,29/11/2023 09:00:00,...,Executado por Muka e Adriano,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento
296,13/11/2023 11:07:00,196257,Rua Alberto Jacobi,LA - Conserto Vazamento (Unidade Nao Identific...,Operacional,NaN,Schroeder I .,SCHROEDER,1001 - Operacional,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vazamento


In [39]:
geolocator = Nominatim(user_agent="myGeocoder")
coordenadas = []


# Iterando pelas linhas do DataFrame e obtendo as coordenadas
for index, row in tqdm(log.iterrows(), total=len(log), desc="Geocodificação"):
    rua = row['Endereço']

    # Primeiro, tenta obter as coordenadas com a rua e número
    if not pd.isnull(rua):
        endereco = f"{rua} - Schroeder - Santa Catarina"
    location = None

    try:
        location = geolocator.geocode(endereco)
        print(location)

        if location:
            coordenadas.append((location.latitude, location.longitude))
        else:
            coordenadas.append((None, None))
            
    except Exception as e:
        coordenadas.append((None, None))

    time.sleep(1)  # Adiciona um pequeno intervalo para evitar bloqueios por uso excessivo da API


# Criando as colunas 'x', 'y', 'Bairro' e 'Cidade' no DataFrame e atribuindo as coordenadas
log['x'] = [coord[0] for coord in coordenadas]
log['y'] = [coord[1] for coord in coordenadas]

# Verificação da cidade: Se a coordenada não tiver como cidade a Palhoça - SC, preencher com NaN
log.loc[log['Cidade'].isna() | ~log['Cidade'].astype(str).str.contains('Schroeder', case=False), ['x', 'y']] = (np.nan, np.nan)

# Criando um objeto transformer para a transformação das coordenadas
transformer = pyproj.Transformer.from_crs("epsg:4326", "epsg:32722", always_xy=True)

# Aplicando a transformação das coordenadas e atribuindo os resultados a 'Latitude' e 'Longitude'.
log['Longitude'], log['Latitude'] = zip(*log.apply(lambda row: transformer.transform(row['y'], row['x']), axis=1))

# Exibindo o DataFrame com as coordenadas transformadas
display(log)

Geocodificação:   0%|                                                                           | 0/55 [00:00<?, ?it/s]

Rua Marechal Castelo Branco, Centro Sul, Schroeder, Região Geográfica Imediata de Joinville, Região Geográfica Intermediária de Joinville, Santa Catarina, Região Sul, 89257-730, Brasil


Geocodificação:   2%|█▏                                                                 | 1/55 [00:02<01:54,  2.12s/it]

Rua Barão do Rio Branco, Centro Leste, Schroeder, Região Geográfica Imediata de Joinville, Região Geográfica Intermediária de Joinville, Santa Catarina, Região Sul, 89275-000, Brasil


Geocodificação:   4%|██▍                                                                | 2/55 [00:03<01:25,  1.60s/it]

...

Rua Independência, Braço do Sul, Schroeder, Região Geográfica Imediata de Joinville, Região Geográfica Intermediária de Joinville, Santa Catarina, Região Sul, 89275-000, Brasil


Geocodificação:  29%|███████████████████▏                                              | 16/55 [00:21<00:51,  1.32s/it]

None


Geocodificação:  29%|███████████████████▏                                              | 16/55 [00:23<00:56,  1.45s/it]


KeyboardInterrupt: 

In [38]:
log['Matrícula'] = np.nan
log['Número'] = np.nan
log['Latitude'] = np.nan
log['Longitude'] = np.nan

log = log[['Matrícula','Categoria de serviço','Data de Solicitação','Endereço','Bairro','Número','Latitude','Longitude']]
log.to_csv("Logradouro tratado.csv",sep=';',encoding = 'latin1')
log

,Matrícula,Categoria de serviço,Data de Solicitação,Endereço,Bairro,Número,Latitude,Longitude
2,NaN,Vazamento,02/01/2023 08:36:00,Rua Mal. Castelo Branco,Bracinho,NaN,NaN,NaN
7,NaN,Vazamento,06/01/2023 08:19:00,Rua Barao Do Rio Branco,Centro Leste,NaN,NaN,NaN
16,NaN,Vazamento,10/01/2023 10:08:00,Rua Guilherme Piske,Centro Norte,NaN,NaN,NaN
17,NaN,Vazamento,10/01/2023 14:36:00,Rua Henrique Ziebel,Rio Hern,NaN,NaN,NaN
25,NaN,Vazamento,16/01/2023 13:33:00,Rua Jorge Lacerda,Centro Norte,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
280,NaN,Vazamento,09/10/2023 08:15:00,Rua Barao Do Rio Branco,Centro Leste,NaN,NaN,NaN
291,NaN,Vazamento,06/11/2023 08:42:00,Rua Otto Elert,Itoupava Acu,NaN,NaN,NaN
292,NaN,Vazamento,06/11/2023 08:47:00,Rua Henrique Ziebel,Rio Hern,NaN,NaN,NaN
296,NaN,Vazamento,13/11/2023 11:07:00,Rua Alberto Jacobi,Schroeder I .,NaN,NaN,NaN
